# Task 6 · English QA
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
checkout = Path(CHECKOUT)
if not checkout.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
subprocess.run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(checkout / PROJECT_SUBDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 1. Download all three English QA splits on Colab
Only `qa/mena/*_en.{jsonl,parquet}` and three `images_mena_*.zip` are selected.
No audio, other languages, or embedded-media demo files.


In [ ]:
%pip -q install huggingface-hub==0.36.0 Pillow==11.3.0
!python -m mmcultureqa.data.prepare download
!python -m mmcultureqa.data.prepare prepare


## 2. Editable rclone settings
Set everything in the next cell. `RCLONE_CONFIG_PATH` points to an existing, authenticated
rclone.conf (upload it via Colab's Files panel, or mount Drive in the optional cell).
`REMOTE` is the section name in that file, without a trailing colon.

Set `ROOT_FOLDER_ID` to a Drive folder ID or its full folder URL. When it is set,
`DESTINATION_SUBDIR` is relative to that folder. Empty means My Drive root, overriding
any root_folder_id already in the supplied config. Example: select the MMCQA folder as
root and use `task6-data/raw` as destination. Without a root, use `MMCQA/task6-data/raw`.
The supplied config is copied to a private temporary file; the original is never modified.
An authenticated config is still required: a folder ID alone does not grant access.


In [ ]:
# Optional: run if rclone.conf is stored on Drive rather than uploaded to /content.
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# Change these settings directly in the notebook.
RCLONE_CONFIG_PATH = '/content/rclone.conf'  # or /content/drive/MyDrive/.../rclone.conf
REMOTE = 'gdrive'
ROOT_FOLDER_ID = ''  # folder ID or https://drive.google.com/drive/folders/...
DESTINATION_SUBDIR = ''  # empty: auto-select task6-data/raw if root set, else MMCQA/task6-data/raw
TRANSFERS = 4
CHECKERS = 8


In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y rclone
import configparser, re, tempfile
from urllib.parse import urlparse

original_config = Path(RCLONE_CONFIG_PATH).expanduser()
if not original_config.is_file():
    raise FileNotFoundError(f'Upload rclone.conf or mount Drive, then set RCLONE_CONFIG_PATH: {original_config}')
sections = configparser.RawConfigParser()
sections.read(original_config, encoding='utf-8-sig')
if REMOTE not in sections or sections[REMOTE].get('type') != 'drive':
    raise ValueError('REMOTE must name a Google Drive section in the supplied rclone.conf')
root_id = ROOT_FOLDER_ID.strip()
if root_id.startswith(('https://', 'http://')):
    match = re.search(r'/folders/([^/?]+)', urlparse(root_id).path)
    if not match:
        raise ValueError('Use a Google Drive folder URL containing /folders/<id>')
    root_id = match.group(1)
if root_id and not re.fullmatch(r'[A-Za-z0-9_-]+', root_id):
    raise ValueError('Invalid Drive folder ID')
subdir = DESTINATION_SUBDIR.strip() or ('task6-data/raw' if root_id else 'MMCQA/task6-data/raw')
if subdir.startswith('/') or ':' in subdir or '..' in Path(subdir).parts:
    raise ValueError('DESTINATION_SUBDIR must be a relative folder path')
if TRANSFERS < 1 or CHECKERS < 1:
    raise ValueError('TRANSFERS and CHECKERS must be positive')

# rclone may refresh tokens; keep writes in a temporary copy, not the original.
temporary_config = tempfile.TemporaryDirectory(prefix='mmcqa-rclone-')
config_path = Path(temporary_config.name) / 'rclone.conf'
config_path.write_bytes(original_config.read_bytes())
config_path.chmod(0o600)
rclone_base = ['rclone', '--config', str(config_path), '--drive-root-folder-id', root_id]
DESTINATION = f'{REMOTE}:{subdir}'
print('Config source:', original_config)
print('Drive root:', root_id or 'My Drive')
print('Destination relative to root:', DESTINATION)
# Read-only check before the upload cell. Credentials are never printed.
subprocess.run(rclone_base + ['lsd', f'{REMOTE}:', '--max-depth', '1'], check=True)


## 3. Upload archives and labels from Colab → Drive
`copy` is incremental and does not delete destination files. Keep archives compressed on Drive;
the experiment notebook copies them to Colab's SSD and extracts locally.


In [ ]:
subprocess.run(rclone_base + ['copy', 'data/raw', DESTINATION,
                '--progress', '--transfers', str(TRANSFERS), '--checkers', str(CHECKERS), '--exclude', '.cache/**',
                '--exclude', 'train/**', '--exclude', 'dev/**', '--exclude', 'devtest/**'], check=True)
subprocess.run(rclone_base + ['check', 'data/raw', DESTINATION,
                '--one-way', '--size-only', '--exclude', '.cache/**', '--exclude', 'train/**',
                '--exclude', 'dev/**', '--exclude', 'devtest/**'], check=True)


In [ ]:
# Remove temporary credentials after the transfer. Reconfigure next session if needed.
temporary_config.cleanup()
